# Stable Diffusion 1.5 — DreamBooth vs LoRA vs Textual Inversion

**Portfolio notebook (revised edition)** · Google Colab / NVIDIA GPU · [Full experiment report](report.pdf)

This notebook organizes the original project around three techniques for teaching Stable Diffusion 1.5 a new subject:

- **DreamBooth**: full UNet fine-tuning on a custom face-shaped plant pot (`mymymy`).
- **LoRA**: parameter-efficient fine-tuning on `mymymy`, including a LoRA adapter for the text encoder.
- **Textual Inversion**: learning an embedding for the public example dog dataset, using token `doSKSg`.

**Historical scope:** The accompanying report also discusses experiments with different dataset sizes, training steps and camera viewpoints. The supplied source notebook contained only the three training runs above, not all of those historical variant runs. The optional sweep *template* below creates **new** experiments; it does not reconstruct missing historical checkpoints.

**Execution:** Run the setup cells on a fresh Colab runtime. Expensive GPU training is disabled by default (`RUN_TRAINING = False`). The notebook will skip inference when an expected checkpoint is not available. The original private training photos and trained model weights are **not** included in the public repository.

**Provenance:** Training scripts are upstream examples from [Hugging Face Diffusers](https://github.com/huggingface/diffusers) and [PEFT](https://github.com/huggingface/peft). Dataset selection, experiment design, prompts, results and analysis are part of this project. Verify the upstream licenses before redistributing scripts or model weights.

## 1. Configuration

All paths are under `/content` in Colab. To train on your own plant-pot photos, either provide a folder in Google Drive, or upload images/ZIP in section 3. Set `RUN_TRAINING = True` **only after checking the dataset paths and GPU**. Use a fresh runtime for a full run.

The dependency versions and training-script tags below are an **explicit, inspectable environment specification**, not a claim that the original January 2026 runs used those exact releases. Colab changes over time, so GPU runs still require real-world validation.

In [ ]:
from pathlib import Path
import os
import sys

IS_COLAB = 'google.colab' in sys.modules
ROOT = Path('/content') if IS_COLAB else Path.cwd()
DATA_ROOT = ROOT / 'data'
RESULTS_ROOT = ROOT / 'results'
MODELS_ROOT = ROOT / 'models'
SOURCES_ROOT = ROOT / 'training_sources'
for p in (DATA_ROOT, RESULTS_ROOT, MODELS_ROOT, SOURCES_ROOT):
    p.mkdir(parents=True, exist_ok=True)

MODEL_ID = 'stable-diffusion-v1-5/stable-diffusion-v1-5'
DOG_REPO = 'diffusers/dog-example'
SEED = 42                     
INFERENCE_STEPS = 50
GUIDANCE_SCALE = 7.5
RESOLUTION = 512
RUN_SETUP = True              
RUN_TRAINING = False        
RUN_INFERENCE = True          
ENABLE_XFORMERS = False      
USE_8BIT_ADAM = True         

POT_DIR = DATA_ROOT / 'mymymy'
DOG_DIR = DATA_ROOT / 'dog'
DB_DIR = MODELS_ROOT / 'dreambooth_mymymy'
LORA_DIR = MODELS_ROOT / 'lora_mymymy'
TI_DIR = MODELS_ROOT / 'textual_inversion_dog'

POT_SOURCE_DIR = None       
UPLOAD_POT_FILES = False      

print('Root:', ROOT)
print('Train:', RUN_TRAINING, '| Inference:', RUN_INFERENCE)

## 2. Environment setup

Install pinned packages **once** per fresh runtime, and obtain matching versions of the upstream training examples. The historical LoRA training code uses the **PEFT `examples/lora_dreambooth` script**, rather than changing the experiment to the newer Diffusers LoRA script (which exposes different adapter settings).

If setup reports a dependency conflict, inspect the Colab Python/PyTorch/CUDA versions and adjust in a fresh runtime; this notebook has not been end-to-end GPU-tested here. `xformers` is optional and deliberately not installed automatically because its wheel must match the Colab PyTorch/CUDA combination.

In [ ]:
import subprocess
import importlib.util
import shlex

def run_checked(args, *, cwd=None):
    args = [str(a) for a in args]
    print('>', shlex.join(args))
    subprocess.run(args, cwd=str(cwd) if cwd else None, check=True)

def clone_tag(url, tag, destination):
    if destination.exists() and (destination / '.git').exists():
        current = subprocess.check_output(
            ['git', '-C', str(destination), 'describe', '--tags', '--exact-match'],
            text=True, stderr=subprocess.DEVNULL
        ).strip() if subprocess.run(
            ['git', '-C', str(destination), 'describe', '--tags', '--exact-match'],
            capture_output=True
        ).returncode == 0 else None
        if current == tag:
            print('Already checked out:', destination, tag)
            return
        raise RuntimeError(f'{destination} exists at a different revision; remove it or use a fresh runtime.')
    if destination.exists():
        raise FileExistsError(f'Folder exists but is not a Git checkout: {destination}')
    run_checked(['git', 'clone', '--depth', '1', '--branch', tag, url, destination])

if RUN_SETUP:
    # Leave torch/torchvision at versions preinstalled in the Colab GPU runtime.
    packages = [
        'diffusers==0.33.1', 'peft==0.15.2', 'transformers==4.51.3',
        'accelerate==1.6.0', 'huggingface_hub==0.30.2',
        'datasets==3.5.1', 'bitsandbytes==0.45.5',
        'ftfy', 'tensorboard', 'safetensors', 'psutil', 'matplotlib', 'Pillow'
    ]
    run_checked([sys.executable, '-m', 'pip', 'install', '-q', *packages])
    clone_tag('https://github.com/huggingface/diffusers', 'v0.33.1', SOURCES_ROOT / 'diffusers')
    clone_tag('https://github.com/huggingface/peft', 'v0.15.2', SOURCES_ROOT / 'peft')

DB_SCRIPT = SOURCES_ROOT / 'diffusers/examples/dreambooth/train_dreambooth.py'
TI_SCRIPT = SOURCES_ROOT / 'diffusers/examples/textual_inversion/textual_inversion.py'
LORA_SCRIPT = SOURCES_ROOT / 'peft/examples/lora_dreambooth/train_dreambooth.py'
if RUN_TRAINING:
    missing = [p for p in (DB_SCRIPT, TI_SCRIPT, LORA_SCRIPT) if not p.is_file()]
    if missing:
        raise FileNotFoundError(f'Training scripts missing: {missing}. Run setup above.')

## 3. Dataset preparation

The original project used the author's **face-shaped plant pot** (`mymymy`), photographed from multiple angles/backgrounds, and the public Hugging Face [`diffusers/dog-example`](https://huggingface.co/datasets/diffusers/dog-example) dataset (`doSKSg`).

**Privacy:** The notebook does not upload or commit personal photos. Avoid pushing the `data/`, `models/` or `results/` folders to GitHub. Training images are normalized to RGB with EXIF orientation corrected. Originals remain unchanged.

In [ ]:
from PIL import Image, ImageOps
import shutil
import zipfile

IMAGE_SUFFIXES = {'.jpg', '.jpeg', '.png', '.webp'}

def image_paths(folder):
    folder = Path(folder)
    if not folder.exists():
        return []
    return sorted(p for p in folder.rglob('*') if p.is_file() and p.suffix.lower() in IMAGE_SUFFIXES)

def prepare_images(source, destination):
    source, destination = Path(source), Path(destination)
    if not source.is_dir():
        raise FileNotFoundError(f'Image directory not found: {source}')
    paths = image_paths(source)
    if not paths:
        raise ValueError(f'No JPG/JPEG/PNG/WEBP images found in: {source}')
    destination.mkdir(parents=True, exist_ok=True)
    # Fresh, unambiguous names; do not remove any user-owned files or directories.
    if image_paths(destination):
        raise FileExistsError(f'{destination} already contains images. Use a fresh output folder.')
    for idx, path in enumerate(paths):
        with Image.open(path) as raw:
            normalized = ImageOps.exif_transpose(raw).convert('RGB')
            normalized.save(destination / f'image_{idx:03d}.jpg', quality=95)
    print(f'Prepared {len(paths)} images in {destination}')
    return destination

def preview_images(folder, title, max_images=20):
    import math
    import matplotlib.pyplot as plt
    paths = image_paths(folder)[:max_images]
    if not paths:
        print(f'No images to preview: {folder}')
        return
    cols = min(5, len(paths))
    rows = math.ceil(len(paths) / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 2.7, rows * 3), squeeze=False)
    for ax in axes.flat:
        ax.axis('off')
    for ax, path in zip(axes.flat, paths):
        with Image.open(path) as im:
            ax.imshow(im.convert('RGB'))
        ax.set_title(path.name, fontsize=8)
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()

In [ ]:
# Option A: use a directory on Drive / local Colab storage.
if POT_SOURCE_DIR:
    source_path = Path(POT_SOURCE_DIR)
    if IS_COLAB and str(source_path).startswith('/content/drive/') and not Path('/content/drive/MyDrive').exists():
        from google.colab import drive
        drive.mount('/content/drive')
    if POT_DIR.resolve() != source_path.resolve():
        if image_paths(POT_DIR):
            print('Using existing prepared pot images:', POT_DIR)
        else:
            prepare_images(source_path, POT_DIR)

# Option B: upload photos or a ZIP in Google Colab.
if UPLOAD_POT_FILES:
    if not IS_COLAB:
        raise RuntimeError('The upload dialog requires Google Colab.')
    from google.colab import files as colab_files
    uploaded = colab_files.upload()
    staging = DATA_ROOT / 'pot_upload_staging'
    staging.mkdir(parents=True, exist_ok=True)
    for name, content in uploaded.items():
        path = Path(name)
        if path.suffix.lower() == '.zip':
            with zipfile.ZipFile(__import__('io').BytesIO(content)) as archive:
                # Reject paths outside the staging directory.
                for entry in archive.infolist():
                    dest = (staging / entry.filename).resolve()
                    if not dest.is_relative_to(staging.resolve()):
                        raise ValueError(f'Unsafe path in ZIP: {entry.filename}')
                archive.extractall(staging)
        elif path.suffix.lower() in IMAGE_SUFFIXES:
            (staging / path.name).write_bytes(content)
        else:
            print('Skipping unsupported file:', name)
    prepare_images(staging, POT_DIR)

preview_images(POT_DIR, 'Custom subject: mymymy')

In [ ]:
DOWNLOAD_DOG_DATA = RUN_TRAINING  # set True to fetch the sample dataset without training
if DOWNLOAD_DOG_DATA:
    from huggingface_hub import snapshot_download
    dog_source = DATA_ROOT / 'dog_source'
    if not image_paths(dog_source):
        snapshot_download(
            repo_id=DOG_REPO, repo_type='dataset',
            local_dir=str(dog_source),
            ignore_patterns=['.gitattributes', '.cache/*'],
        )
    if not image_paths(DOG_DIR):
        prepare_images(dog_source, DOG_DIR)

preview_images(DOG_DIR, 'Reference subject: doSKSg dog')

## 4. Training helpers and experiment settings

These settings mirror the training values visible in the supplied notebook and report, **except** where explicitly noted:

| Setting | DreamBooth | LoRA (PEFT) | Textual Inversion |
|---|---:|---:|---:|
| Subject | `mymymy` | `mymymy` | `doSKSg` (dog) |
| Steps | 400 | 400 | 400 |
| Learning rate argument | `5e-6` | `1e-4` | `5e-4` |
| Batch size | 1 | 1 | 1 |
| Accumulation | 1 | 1 | 4 |
| Resolution | 512 | 512 | 512 |
| Train text encoder | No | Yes (LoRA adapter) | Learns token embedding |
| LoRA rank / alpha | — | UNet `16 / 27`, text encoder `16 / 17` | — |

**Important Textual Inversion detail:** the original script used `--scale_lr`. On a single GPU, the provided `5e-4` is multiplied by accumulation `4` and batch size `1` (effective learning rate approximately `2e-3`). This is **preserved** below and is not identical to running with `5e-4` without scaling.

**Changes:** the DreamBooth prompt is now passed as the actual string, seed and explicit mixed precision are configured, `xformers` is optional, and paths/output naming are normalized. Consequently new model weights need not be numerically identical to the historic results.

In [ ]:
import torch
import gc
import json

DB_PROMPT = 'a photo of mymymy face pot from different sides and layers'
LORA_PROMPT = 'a photo of mymymy face pot'
TI_PLACEHOLDER = 'doSKSg'

def ensure_gpu():
    if not torch.cuda.is_available():
        raise RuntimeError('A CUDA GPU is required. In Colab: Runtime > Change runtime type > GPU.')
    print('GPU:', torch.cuda.get_device_name(0))

def ensure_training_dataset(folder, name):
    n = len(image_paths(folder))
    if n == 0:
        raise FileNotFoundError(f'{name}: no training images in {folder}. Prepare the dataset first.')
    print(f'{name}: {n} images')
    return n

def accelerate_train(script, args):
    ensure_gpu()
    if not script.is_file():
        raise FileNotFoundError(f'Install the matching training script: {script}')
    command = ['accelerate', 'launch', '--num_processes=1', '--mixed_precision=fp16', str(script), *args]
    run_checked(command, cwd=ROOT)

def save_experiment_info(method, path, config):
    path.mkdir(parents=True, exist_ok=True)
    (path / 'experiment_config.json').write_text(
        json.dumps({'method': method, 'model': MODEL_ID, 'seed': SEED, **config}, indent=2),
        encoding='utf-8'
    )

def clear_cuda_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

## 5. DreamBooth — custom plant pot

The original code accidentally passed the literal string `prompt` rather than the Python variable into `--instance_prompt`. That is corrected here. The script trains UNet only, so inference loads the **saved pipeline** (no unconditional attempt to load a separately trained text encoder).

In [ ]:
if RUN_TRAINING:
    ensure_training_dataset(POT_DIR, 'DreamBooth')
    db_args = [
        f'--pretrained_model_name_or_path={MODEL_ID}',
        f'--instance_data_dir={POT_DIR}',
        f'--instance_prompt={DB_PROMPT}',
        f'--output_dir={DB_DIR}',
        f'--resolution={RESOLUTION}',
        '--train_batch_size=1', '--gradient_accumulation_steps=1',
        '--learning_rate=5e-6', '--lr_scheduler=constant',
        '--lr_warmup_steps=0', '--max_train_steps=400',
        '--gradient_checkpointing', '--set_grads_to_none',
        f'--seed={SEED}',
    ]
    if USE_8BIT_ADAM:
        db_args.append('--use_8bit_adam')
    if ENABLE_XFORMERS:
        db_args.append('--enable_xformers_memory_efficient_attention')
    accelerate_train(DB_SCRIPT, db_args)
    save_experiment_info('dreambooth', DB_DIR, {
        'instance_prompt': DB_PROMPT, 'max_train_steps': 400,
        'learning_rate': 5e-6, 'train_text_encoder': False
    })
else:
    print('DreamBooth training skipped. Enable RUN_TRAINING to launch.')

## 6. LoRA — custom plant pot (PEFT example)

We retain the original **PEFT DreamBooth-LoRA** training script, including distinct LoRA settings for UNet and CLIP text encoder. This is intentional: silently changing to the Diffusers LoRA training script would change the original method and its alpha settings.

In [ ]:
def train_lora(dataset_dir=POT_DIR, output_dir=LORA_DIR, steps=400, seed=SEED):
    ensure_training_dataset(dataset_dir, 'LoRA')
    args = [
        f'--pretrained_model_name_or_path={MODEL_ID}',
        f'--instance_data_dir={dataset_dir}',
        f'--instance_prompt={LORA_PROMPT}',
        f'--output_dir={output_dir}',
        '--train_text_encoder', '--num_dataloader_workers=1',
        f'--resolution={RESOLUTION}', '--train_batch_size=1',
        '--lr_scheduler=constant', '--lr_warmup_steps=0',
        '--use_lora', '--lora_r=16', '--lora_alpha=27',
        '--lora_text_encoder_r=16', '--lora_text_encoder_alpha=17',
        '--learning_rate=1e-4', '--gradient_accumulation_steps=1',
        '--gradient_checkpointing', f'--max_train_steps={steps}', f'--seed={seed}',
    ]
    if ENABLE_XFORMERS:
        args.append('--enable_xformers_memory_efficient_attention')
    accelerate_train(LORA_SCRIPT, args)
    save_experiment_info('peft_lora', Path(output_dir), {
        'instance_prompt': LORA_PROMPT, 'max_train_steps': steps,
        'learning_rate': 1e-4, 'unet_lora_rank': 16,
        'unet_lora_alpha': 27, 'text_encoder_lora_rank': 16,
        'text_encoder_lora_alpha': 17, 'train_text_encoder': True,
        'dataset_images': len(image_paths(dataset_dir))
    })

if RUN_TRAINING:
    train_lora()
else:
    print('LoRA training skipped. Enable RUN_TRAINING to launch.')

## 7. Textual Inversion — `doSKSg` dog

Trains a learned embedding for a new token. The output is an embedding, **not** a full Stable Diffusion checkpoint. The original `--scale_lr` option is preserved; see the learning-rate note above.

In [ ]:
if RUN_TRAINING:
    ensure_training_dataset(DOG_DIR, 'Textual Inversion')
    ti_args = [
        f'--pretrained_model_name_or_path={MODEL_ID}',
        f'--train_data_dir={DOG_DIR}',
        '--learnable_property=object',
        f'--placeholder_token={TI_PLACEHOLDER}',
        '--initializer_token=dog',
        f'--resolution={RESOLUTION}', '--train_batch_size=1',
        '--gradient_accumulation_steps=4', '--max_train_steps=400',
        '--learning_rate=5e-4', '--scale_lr',
        '--lr_scheduler=constant', '--lr_warmup_steps=0',
        f'--output_dir={TI_DIR}', f'--seed={SEED}',
    ]
    if ENABLE_XFORMERS:
        ti_args.append('--enable_xformers_memory_efficient_attention')
    accelerate_train(TI_SCRIPT, ti_args)
    save_experiment_info('textual_inversion', TI_DIR, {
        'placeholder_token': TI_PLACEHOLDER, 'initializer_token': 'dog',
        'max_train_steps': 400, 'learning_rate_argument': 5e-4,
        'scale_lr': True, 'gradient_accumulation_steps': 4,
        'effective_lr_single_gpu': 2e-3
    })
else:
    print('Textual Inversion training skipped. Enable RUN_TRAINING to launch.')

## 8. Inference and reproducible comparisons

Compare **base Stable Diffusion 1.5** against each trained variant using identical prompts, inference steps, guidance scale and seeded generators. Pipelines are loaded **one at a time** to avoid excessive GPU memory use. Outputs are placed in a method-specific subfolder under `results/`.

Two prompt families below come from the original notebook/report: generalization to new settings and view/orientation. Additional exploratory prompts from the original notebook are kept in a separate variable rather than being used by default.

In [ ]:
GENERAL_PROMPTS_POT = [
    'A picture of a mymymy face pot on a house roof',
    'A picture of a mymymy face pot in crochet style',
    'A picture of a mymymy face pot on a football pitch',
    'A picture of a mymymy face pot in a pool underwater',
]
GENERAL_PROMPTS_DOG = [
    'A picture of a doSKSg dog on a house roof',
    'A picture of a doSKSg dog in crochet style',
    'A picture of a doSKSg dog on a football pitch',
    'A picture of a doSKSg dog in a pool underwater',
]
ORIENTATION_PROMPTS_POT = [
    'The view from the top of mymymy face pot',
    'The view from the bottom of mymymy face pot',
    'The view of mymymy face pot hidden behind a dragon',
    'The back view of mymymy face pot',
]
ORIGINAL_EXPLORATORY_PROMPTS_POT = [
    'A picture of a mymymy face pot without clothes',
    'A picture of a mymymy face pot in bikini',
    'A picture of a mymymy face pot killing potato',
    "A picture of a mymymy face pot in Julia's arms",
]

from diffusers import StableDiffusionPipeline
from peft import PeftModel

def base_pipeline():
    return StableDiffusionPipeline.from_pretrained(MODEL_ID, torch_dtype=torch.float16)

def dreambooth_pipeline():
    # Upstream DreamBooth script saves a full Diffusers pipeline.
    return StableDiffusionPipeline.from_pretrained(DB_DIR, torch_dtype=torch.float16)

def peft_lora_pipeline():
    pipe = base_pipeline()
    unet_path = LORA_DIR / 'unet'
    text_encoder_path = LORA_DIR / 'text_encoder'
    if not (unet_path / 'adapter_config.json').is_file():
        raise FileNotFoundError(f'Missing PEFT UNet adapter: {unet_path}')
    pipe.unet = PeftModel.from_pretrained(pipe.unet, str(unet_path))
    if (text_encoder_path / 'adapter_config.json').is_file():
        pipe.text_encoder = PeftModel.from_pretrained(pipe.text_encoder, str(text_encoder_path))
    return pipe

def textual_inversion_pipeline():
    pipe = base_pipeline()
    embeddings = [TI_DIR / 'learned_embeds.safetensors', TI_DIR / 'learned_embeds.bin']
    existing = next((p for p in embeddings if p.exists()), None)
    if existing is None:
        raise FileNotFoundError(f'No learned textual inversion embedding in {TI_DIR}')
    pipe.load_textual_inversion(str(existing), token=TI_PLACEHOLDER)
    return pipe

def is_checkpoint_ready(method):
    if method == 'dreambooth':
        return (DB_DIR / 'model_index.json').is_file()
    if method == 'lora':
        return (LORA_DIR / 'unet/adapter_config.json').is_file()
    if method == 'textual_inversion':
        return any((TI_DIR / name).is_file() for name in ('learned_embeds.bin', 'learned_embeds.safetensors'))
    raise ValueError(method)

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

def generate_batch(factory, prompts, folder, prefix, seed=0):
    ensure_gpu()
    folder.mkdir(parents=True, exist_ok=True)
    pipe = factory().to('cuda')
    try:
        for i, prompt in enumerate(prompts):
            generator = torch.Generator(device='cuda').manual_seed(seed)
            image = pipe(
                prompt, num_inference_steps=INFERENCE_STEPS,
                guidance_scale=GUIDANCE_SCALE, generator=generator,
            ).images[0]
            image.save(folder / f'{prefix}_{i:02d}.png')
            print(f'{prefix}_{i:02d}: {prompt}')
    finally:
        del pipe
        clear_cuda_memory()

def show_comparison(folder, prompts, title):
    base_images = [folder / f'base_{i:02d}.png' for i in range(len(prompts))]
    tuned_images = [folder / f'tuned_{i:02d}.png' for i in range(len(prompts))]
    if not all(path.is_file() for path in base_images + tuned_images):
        print('Comparison skipped; some output images are missing:', folder)
        return
    fig, axes = plt.subplots(2, len(prompts), figsize=(len(prompts) * 3.2, 7), squeeze=False)
    for i, prompt in enumerate(prompts):
        for row, path in enumerate((base_images[i], tuned_images[i])):
            with Image.open(path) as image:
                axes[row, i].imshow(image.convert('RGB'))
            axes[row, i].axis('off')
        axes[0, i].set_title(prompt, fontsize=8, wrap=True)
    axes[0, 0].set_ylabel('Base', fontsize=12)
    axes[1, 0].set_ylabel('Fine-tuned', fontsize=12)
    fig.suptitle(title)
    plt.tight_layout()
    fig.savefig(folder / 'comparison.png', dpi=150, bbox_inches='tight')
    plt.show()


def compare_models(method, prompts, tuned_factory, seed=0):
    folder = RESULTS_ROOT / method
    if RUN_INFERENCE:
        if is_checkpoint_ready(method):
            generate_batch(base_pipeline, prompts, folder, 'base', seed)
            generate_batch(tuned_factory, prompts, folder, 'tuned', seed)
        else:
            print(f'{method}: checkpoint not present; generation skipped.')
    show_comparison(folder, prompts, f'Base SD 1.5 vs {method}')

In [ ]:
# DreamBooth: generalization prompts from the report
compare_models('dreambooth', GENERAL_PROMPTS_POT, dreambooth_pipeline, seed=0)

In [ ]:
# LoRA: original notebook focused on the orientation prompt set.
# To compare the four settings from the report, use GENERAL_PROMPTS_POT instead.
compare_models('lora', ORIENTATION_PROMPTS_POT, peft_lora_pipeline, seed=10)

In [ ]:
# Textual Inversion: public dog reference dataset
compare_models('textual_inversion', GENERAL_PROMPTS_DOG, textual_inversion_pipeline, seed=0)

## 9. Optional: fresh LoRA sweeps (not previously archived results)

The report analyses `50/200/1000` training steps and subsets of `3/10/14` plant photos, but those distinct training invocations were **not included** in the supplied notebook. The following is a clearly labeled **template for new training runs**. Choosing the first `n` images is deterministic, but **may not match** the actual historical subset selection; do not label new results as a reproduction of the originals without verifying the photo lists.

Requires enough GPU time/VRAM and the custom dataset. These sweeps are **disabled** by default.

In [ ]:
RUN_NEW_LORA_SWEEPS = False
NEW_LORA_STEPS = [50, 200, 1000]         # new experiments, do not overwrite existing models
NEW_LORA_DATASET_SIZES = [3, 10, 14]

def make_subset(source, count, destination):
    paths = image_paths(source)
    if len(paths) < count:
        raise ValueError(f'Need {count} images; found {len(paths)} in {source}')
    destination.mkdir(parents=True, exist_ok=True)
    if image_paths(destination):
        raise FileExistsError(f'Subset folder already contains images: {destination}')
    for src in paths[:count]:
        shutil.copy2(src, destination / src.name)
    return destination

if RUN_NEW_LORA_SWEEPS:
    ensure_training_dataset(POT_DIR, 'LoRA sweep source')
    for step_count in NEW_LORA_STEPS:
        output = MODELS_ROOT / f'lora_new_steps_{step_count}'
        train_lora(POT_DIR, output, steps=step_count)
    for image_count in NEW_LORA_DATASET_SIZES:
        subset_dir = DATA_ROOT / f'pot_subset_first_{image_count}'
        make_subset(POT_DIR, image_count, subset_dir)
        output = MODELS_ROOT / f'lora_new_images_{image_count}'
        train_lora(subset_dir, output, steps=400)
else:
    print('Optional new LoRA sweeps skipped.')

## 10. Export the generated images

Exports only generated comparison images and experiment metadata (not private training photos or large model weights). The ZIP is created only when such files exist.

In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED

export_items = [p for p in RESULTS_ROOT.rglob('*') if p.is_file()]
if export_items:
    archive_path = ROOT / 'sd_finetuning_comparison_results.zip'
    with ZipFile(archive_path, 'w', ZIP_DEFLATED) as z:
        for path in export_items:
            z.write(path, arcname=str(path.relative_to(RESULTS_ROOT.parent)))
    print('Created:', archive_path, 'files:', len(export_items))
    # Optional download in Google Colab:
    # from google.colab import files as colab_files
    # colab_files.download(str(archive_path))
else:
    print('No generated results to export yet.')

## 11. Interpretation and limits

**Observations from the original report** (not re-evaluated automatically by this notebook):

- DreamBooth produced the most faithful images of the introduced subject in the author’s qualitative comparison.
- LoRA and Textual Inversion produced useful results with fewer trainable parameters, but could miss important subject characteristics for smaller/shorter runs.
- Longer LoRA training improved subject fidelity but the report describes overfitting at 1,000 steps, including weaker adherence to background/style instructions.
- For the orientation experiment, the report uses **three separately constructed datasets**; those image subsets and individual training commands were not present in the uploaded notebook.